---
## Module 4 - Deep Learning

**Deep Learning** uses many-layered networks to automatically learn hierarchical feature representations.

| Concept | Key Idea |
|---------|----------|
| Deep Neural Networks (DNN) | Many hidden layers, automatic feature extraction |
| Transfer Learning | Reuse pretrained weights for new tasks |
| GANs | Generator + Discriminator adversarial training |
| Attention Mechanisms | Selectively focus on relevant input parts |
| Deep RL | Combine deep networks with reward signals |
| Dropout | Regularization by randomly zeroing activations |
| Capsule Networks | Preserves spatial hierarchies in data |


### 4.1 Deep Neural Network Architecture - Visual Depth Analysis

In [ ]:
from tensorflow.keras.datasets import cifar10
(Xc_tr, yc_tr), (Xc_te, yc_te) = cifar10.load_data()
Xc_tr = Xc_tr.astype('float32')/255.0; Xc_te = Xc_te.astype('float32')/255.0
Yc_tr = to_categorical(yc_tr, 10);      Yc_te = to_categorical(yc_te, 10)
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Show sample images
fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
for i in range(10):
    for j, (X_show, y_show) in enumerate([(Xc_tr, yc_tr), (Xc_te, yc_te)]):
        idx = np.where(y_show.ravel() == i)[0][0]
        axes[j, i].imshow(X_show[idx]); axes[j, i].axis('off')
        axes[j, i].set_title(class_names[i], fontsize=7)
plt.suptitle('CIFAR-10 Dataset  Sample Images', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_cifar10_samples.png', dpi=130, bbox_inches='tight')
plt.show()

# Flatten once up front
Xc_tr_flat = Xc_tr.reshape(len(Xc_tr), -1)   # (50000, 3072)
Xc_te_flat = Xc_te.reshape(len(Xc_te), -1)   # (10000, 3072)

def build_dnn(depth, width=128):
    m = keras.Sequential(name=f'DNN_depth{depth}')
    m.add(layers.Dense(width, activation='relu', input_shape=(3072,)))
    m.add(layers.BatchNormalization())
    m.add(layers.Dropout(0.3))
    for _ in range(depth - 1):
        m.add(layers.Dense(width, activation='relu'))
        m.add(layers.BatchNormalization())
        m.add(layers.Dropout(0.3))
    m.add(layers.Dense(10, activation='softmax'))
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

depth_results = []
for depth in [1, 2, 4]:
    m = build_dnn(depth)
    h = m.fit(Xc_tr_flat, Yc_tr, epochs=8, batch_size=512,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(Xc_te_flat, Yc_te, verbose=0)
    depth_results.append({'Depth': depth, 'Params': m.count_params(),
                           'Test Acc': f'{acc*100:.2f}%'})

df_depth = pd.DataFrame(depth_results)
print("=== Effect of Network Depth on CIFAR-10 ===")
print(df_depth.to_string(index=False))

### 4.2 Transfer Learning - MobileNetV2 on CIFAR-10

In [ ]:
#  Fine-tune MobileNetV2 pretrained on ImageNet 
import tensorflow as tf
(Xc_tr, yc_tr), (Xc_te, yc_te) = cifar10.load_data()

# Resize to 96x96 for MobileNetV2 minimum input
def preprocess(X):
    X = tf.image.resize(X, [96, 96])
    return tf.keras.applications.mobilenet_v2.preprocess_input(X)

# Use small subset for speed
N_TRAIN, N_TEST = 5000, 1000
X_tl_tr = preprocess(Xc_tr[:N_TRAIN].astype('float32'))
X_tl_te = preprocess(Xc_te[:N_TEST].astype('float32'))
Y_tl_tr = to_categorical(yc_tr[:N_TRAIN], 10)
Y_tl_te = to_categorical(yc_te[:N_TEST],  10)

# Feature extraction (frozen base)
base = tf.keras.applications.MobileNetV2(input_shape=(96,96,3),
                                          include_top=False, weights='imagenet')
base.trainable = False

tl_model = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax'),
], name='TransferLearning_MobileNetV2')

tl_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
tl_model.summary()

h_tl = tl_model.fit(X_tl_tr, Y_tl_tr, epochs=10, batch_size=64,
                     validation_split=0.1, verbose=1)
_, tl_acc = tl_model.evaluate(X_tl_te, Y_tl_te, verbose=0)
print(f"\n Transfer Learning Test Accuracy: {tl_acc*100:.2f}% (trained on only {N_TRAIN} samples!)")

fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(h_tl.history['accuracy'],     label='Train', lw=2)
axes[0].plot(h_tl.history['val_accuracy'], label='Val',   lw=2)
axes[0].set_title('Transfer Learning Accuracy', fontweight='bold'); axes[0].legend()
axes[1].plot(h_tl.history['loss'],     label='Train', lw=2, color='orange')
axes[1].plot(h_tl.history['val_loss'], label='Val',   lw=2, color='red')
axes[1].set_title('Transfer Learning Loss', fontweight='bold'); axes[1].legend()
plt.suptitle(f'MobileNetV2 Transfer Learning - Acc: {tl_acc*100:.2f}%', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/04_transfer_learning.png', dpi=130, bbox_inches='tight'); plt.show()

### 4.3 Generative Adversarial Network (GAN) - MNIST Digit Generation

In [ ]:
#  Simple GAN on MNIST 
LATENT_DIM = 100
IMG_DIM     = 784

def build_generator(latent_dim):
    return keras.Sequential([
        layers.Dense(256, input_shape=(latent_dim,)),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Dense(512),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Dense(1024),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Dense(IMG_DIM, activation='tanh'),
    ], name='Generator')

def build_discriminator():
    return keras.Sequential([
        layers.Dense(512, input_shape=(IMG_DIM,)),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Dense(256),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ], name='Discriminator')

generator     = build_generator(LATENT_DIM)
discriminator = build_discriminator()
discriminator.compile(optimizer=keras.optimizers.Adam(0.0002, 0.5),
                      loss='binary_crossentropy', metrics=['accuracy'])

z     = layers.Input(shape=(LATENT_DIM,))
img   = generator(z)
discriminator.trainable = False
valid = discriminator(img)
gan   = keras.Model(z, valid, name='GAN')
gan.compile(optimizer=keras.optimizers.Adam(0.0002, 0.5), loss='binary_crossentropy')

(X_gan, _), _ = mnist.load_data()
X_gan = (X_gan.astype('float32') / 127.5) - 1.0
X_gan = X_gan.reshape(-1, 784)

BATCH, EPOCHS = 128, 100
d_losses, g_losses = [], []
sample_noise = np.random.normal(0, 1, (25, LATENT_DIM))

BATCH, EPOCHS = 256, 50
d_losses, g_losses = [], []
sample_noise = np.random.normal(0, 1, (25, LATENT_DIM))
steps = X_gan.shape[0] // BATCH  # ~234 steps per epoch

for epoch in range(EPOCHS):
    d_epoch, g_epoch = [], []
    # Shuffle data each epoch
    np.random.shuffle(X_gan)
    for step in range(steps):
        real_imgs = X_gan[step*BATCH:(step+1)*BATCH]
        noise     = np.random.normal(0, 1, (BATCH, LATENT_DIM))
        fake_imgs = generator(noise, training=False).numpy()
        real_lbl  = np.ones((BATCH, 1)) * 0.9
        fake_lbl  = np.zeros((BATCH, 1))
        discriminator.trainable = True
        d_loss_r  = discriminator.train_on_batch(real_imgs, real_lbl)
        d_loss_f  = discriminator.train_on_batch(fake_imgs, fake_lbl)
        d_loss    = 0.5 * (d_loss_r[0] + d_loss_f[0])
        noise2    = np.random.normal(0, 1, (BATCH, LATENT_DIM))
        discriminator.trainable = False
        g_loss    = gan.train_on_batch(noise2, np.ones((BATCH, 1)))
        d_epoch.append(d_loss); g_epoch.append(g_loss)
    d_losses.append(np.mean(d_epoch)); g_losses.append(np.mean(g_epoch))
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | D: {d_losses[-1]:.4f} | G: {g_losses[-1]:.4f}")

generated = generator.predict(sample_noise, verbose=0)
generated = (generated + 1) / 2.0

fig, axes = plt.subplots(5, 5, figsize=(8, 8))
fig.patch.set_facecolor('black')
for ax, img in zip(axes.ravel(), generated):
    ax.imshow(img.reshape(28, 28), cmap='gray_r'); ax.axis('off')
plt.suptitle(f'GAN Generated MNIST Digits  Epoch {EPOCHS}',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_gan_generated.png', dpi=130, bbox_inches='tight', facecolor='black')
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(d_losses, label='Discriminator Loss', lw=1.5, color='#E53935')
ax2.plot(g_losses, label='Generator Loss',     lw=1.5, color='#1E88E5')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('GAN Training  Loss Curves', fontweight='bold'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_gan_loss.png', dpi=130, bbox_inches='tight')
plt.show()

### 4.4 Attention Mechanism - Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores  = np.dot(Q, K.T) / np.sqrt(d_k)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)  # softmax
    output  = np.dot(weights, V)
    return output, weights

np.random.seed(0)
SEQ_LEN, D_MODEL = 8, 16
Q = np.random.randn(SEQ_LEN, D_MODEL)
K = np.random.randn(SEQ_LEN, D_MODEL)
V = np.random.randn(SEQ_LEN, D_MODEL)
output, attn_weights = scaled_dot_product_attention(Q, K, V)

tokens = [f'tok_{i}' for i in range(SEQ_LEN)]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(attn_weights, ax=axes[0], cmap='viridis', annot=True, fmt='.2f',
            xticklabels=tokens, yticklabels=tokens)
axes[0].set_title('Attention Weight Matrix\n(how much each token attends to others)',
                   fontweight='bold')
axes[0].set_xlabel('Key tokens'); axes[0].set_ylabel('Query tokens')

im = axes[1].imshow(output, aspect='auto', cmap='plasma')
axes[1].set_xlabel('Embedding Dimension'); axes[1].set_ylabel('Sequence Position')
axes[1].set_yticks(range(SEQ_LEN)); axes[1].set_yticklabels(tokens)
axes[1].set_title('Attention Output\n(weighted value vectors)', fontweight='bold')
plt.colorbar(im, ax=axes[1])

plt.suptitle('Scaled Dot-Product Attention - Core of the Transformer', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/04_attention.png', dpi=130, bbox_inches='tight'); plt.show()

print(f"Query shape:   {Q.shape}")
print(f"Key shape:     {K.shape}")
print(f"Value shape:   {V.shape}")
print(f"Output shape:  {output.shape}")
print(f"Attention max weights per query: {attn_weights.max(axis=1).round(3)}")

### 4.5 Dropout - Regularization Effect

In [ ]:
#  Compare no dropout vs dropout on MNIST 
(X_tr, y_tr), (X_te, y_te) = mnist.load_data()
X_tr_f = X_tr.reshape(-1,784).astype('float32')/255.0
X_te_f = X_te.reshape(-1,784).astype('float32')/255.0
Y_tr_c = to_categorical(y_tr,10); Y_te_c = to_categorical(y_te,10)

results_drop = {}
for name, drop_rate in [('No Dropout', 0.0), ('Dropout=0.3', 0.3), ('Dropout=0.5', 0.5)]:
    m = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(784,)),
        layers.Dropout(drop_rate),
        layers.Dense(256, activation='relu'),
        layers.Dropout(drop_rate),
        layers.Dense(10, activation='softmax'),
    ])
    m.compile('adam','categorical_crossentropy',metrics=['accuracy'])
    h = m.fit(X_tr_f, Y_tr_c, epochs=15, batch_size=256, validation_split=0.1, verbose=0)
    _, te_acc = m.evaluate(X_te_f, Y_te_c, verbose=0)
    results_drop[name] = (h.history['accuracy'], h.history['val_accuracy'], te_acc)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
colors_d = ['#E53935','#1E88E5','#43A047']
for (name, (tr_h, vl_h, te_a)), col in zip(results_drop.items(), colors_d):
    axes[0].plot(tr_h, lw=2, color=col, label=f'{name} (train)')
    axes[0].plot(vl_h, lw=2, color=col, linestyle='--', label=f'{name} (val)')
axes[0].set_title('Dropout - Training vs Validation Accuracy', fontweight='bold'); axes[0].legend(fontsize=8)

names_d  = list(results_drop.keys())
te_accs  = [v[2] for v in results_drop.values()]
overfit  = [max(v[0]) - v[2] for v in results_drop.values()]
x_pos = np.arange(len(names_d)); width=0.35
axes[1].bar(x_pos-width/2, te_accs, width, label='Test Accuracy',   color='#42A5F5')
axes[1].bar(x_pos+width/2, overfit, width, label='Overfit Gap', color='#EF5350')
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(names_d, rotation=10)
axes[1].set_title('Test Accuracy vs Overfit Gap', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/04_dropout.png', dpi=130, bbox_inches='tight'); plt.show()

df_drop = pd.DataFrame({'Model':names_d,'Test Acc':[f'{a:.3f}' for a in te_accs],
                         'Overfit Gap':[f'{g:.4f}' for g in overfit]})
print(df_drop.to_string(index=False))

### 4.6 Deep Reinforcement Learning - Q-Learning on GridWorld

In [ ]:
#  Tabular Q-Learning on a simple GridWorld 
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.goal  = (size-1, size-1)
        self.traps = [(1,1),(2,3),(3,1),(1,3)]
        self.reset()
    def reset(self):
        self.pos = (0,0); return self._state()
    def _state(self): return self.pos[0]*self.size + self.pos[1]
    def step(self, action):
        moves = {0:(-1,0),1:(1,0),2:(0,-1),3:(0,1)}  # up,down,left,right
        dr, dc = moves[action]
        nr = max(0, min(self.size-1, self.pos[0]+dr))
        nc = max(0, min(self.size-1, self.pos[1]+dc))
        self.pos = (nr, nc)
        if self.pos == self.goal:   return self._state(), +10, True
        if self.pos in self.traps:  return self._state(), -5,  True
        return self._state(), -0.1, False

n_states, n_actions = 25, 4
Q = np.zeros((n_states, n_actions))
env = GridWorld(5)
eps, alpha, gamma = 1.0, 0.1, 0.95
episode_rewards = []

for ep in range(2000):
    state = env.reset(); total_r = 0; done = False
    while not done:
        if np.random.rand() < eps:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])
        next_state, reward, done = env.step(action)
        Q[state, action] += alpha * (reward + gamma*np.max(Q[next_state]) - Q[state,action])
        state = next_state; total_r += reward
    episode_rewards.append(total_r)
    if eps > 0.05: eps *= 0.998

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Smoothed reward curve
window = 50
smoothed = np.convolve(episode_rewards, np.ones(window)/window, 'valid')
axes[0].plot(episode_rewards, alpha=0.25, color='gray', lw=0.8)
axes[0].plot(range(window-1, len(episode_rewards)), smoothed, 'b-', lw=2)
axes[0].axhline(0, color='red', linestyle='--', lw=1)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward')
axes[0].set_title('Q-Learning - Reward per Episode', fontweight='bold')

# Learned policy grid
policy_arrows = {0:'',1:'',2:'',3:''}
grid = np.max(Q, axis=1).reshape(5,5)
im = axes[1].imshow(grid, cmap='RdYlGn', alpha=0.7)
plt.colorbar(im, ax=axes[1], label='Max Q-value')
for r in range(5):
    for c in range(5):
        s = r*5+c
        if (r,c) == (4,4):    axes[1].text(c,r,'',ha='center',va='center',fontsize=14)
        elif (r,c) in [(1,1),(2,3),(3,1),(1,3)]:
                              axes[1].text(c,r,'',ha='center',va='center',fontsize=11)
        else:                 axes[1].text(c,r,policy_arrows[np.argmax(Q[s])],
                                           ha='center',va='center',fontsize=14,fontweight='bold')
axes[1].set_title('Learned Policy (arrows = best action)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/04_qlearning.png', dpi=130, bbox_inches='tight'); plt.show()
print(f"Final avg reward (last 100 episodes): {np.mean(episode_rewards[-100:]):.2f}")

### 4.x Capsule Networks  Routing by Agreement on MNIST
Capsule Networks (Sabour et al., 2017) replace scalar activations with
vectors (capsules) that encode both the presence and pose of a feature.
Dynamic routing-by-agreement replaces max-pooling.

In [ ]:
#  Simplified CapsNet on MNIST 
import tensorflow as tf
from tensorflow.keras import layers, models

#  Squash activation (capsule non-linearity) 
def squash(vectors, axis=-1):
    s_sq_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale     = s_sq_norm / (1 + s_sq_norm) / tf.sqrt(s_sq_norm + 1e-8)
    return scale * vectors

class CapsuleLayer(layers.Layer):
    def __init__(self, n_caps, caps_dim, routings=3, **kwargs):
        super().__init__(**kwargs)
        self.n_caps  = n_caps
        self.caps_dim = caps_dim
        self.routings = routings

    def build(self, input_shape):
        self.n_in_caps  = input_shape[1]
        self.in_caps_dim = input_shape[2]
        self.W = self.add_weight(
            shape=(1, self.n_in_caps, self.n_caps, self.caps_dim, self.in_caps_dim),
            initializer='glorot_uniform', trainable=True, name='W')

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        inputs_hat = tf.squeeze(
            tf.matmul(tf.tile(self.W, [batch,1,1,1,1]),
                      tf.expand_dims(tf.expand_dims(inputs, 2), -1)),
            axis=-1)                    # (batch, n_in, n_caps, caps_dim)
        b = tf.zeros_like(inputs_hat[:,:,:,0])   # routing logits
        for i in range(self.routings):
            c = tf.nn.softmax(b, axis=2)
            s = tf.reduce_sum(tf.multiply(tf.expand_dims(c,-1), inputs_hat), axis=1)
            v = squash(s)
            if i < self.routings - 1:
                b += tf.reduce_sum(tf.multiply(
                    tf.expand_dims(v, 1), inputs_hat), axis=-1)
        return v  # (batch, n_caps, caps_dim)

def capsule_length(x):
    return tf.sqrt(tf.reduce_sum(tf.square(x), axis=-1) + 1e-8)

#  Build model 
inp_c  = layers.Input(shape=(28, 28, 1))
conv1  = layers.Conv2D(64, 9, activation='relu', padding='valid')(inp_c)
conv2  = layers.Conv2D(128, 9, strides=2, activation='relu', padding='valid')(conv1)

# Reshape to primary capsules
n_primary, primary_dim = 32, 8
primary_shape = conv2.shape[1] * conv2.shape[2] * (128 // primary_dim)
primary = layers.Reshape((primary_shape, primary_dim))(conv2)
primary = layers.Lambda(squash)(primary)

# Digit capsules
digit_caps = CapsuleLayer(10, 16, routings=3, name='digit_caps')(primary)
out_caps   = layers.Lambda(capsule_length, name='capsule_length')(digit_caps)

capsnet = models.Model(inp_c, out_caps, name='SimpleCapsNet')
capsnet.compile(optimizer=keras.optimizers.Adam(0.001),
                loss='mse', metrics=['accuracy'])
capsnet.summary()

#  Load & train 
(X_tr_c, y_tr_c), (X_te_c, y_te_c) = mnist.load_data()
X_tr_c = X_tr_c.reshape(-1,28,28,1).astype('float32')/255.0
X_te_c = X_te_c.reshape(-1,28,28,1).astype('float32')/255.0
Y_tr_c = to_categorical(y_tr_c, 10)
Y_te_c = to_categorical(y_te_c, 10)

# Use a subset for speed
N = 10000
h_caps = capsnet.fit(X_tr_c[:N], Y_tr_c[:N], epochs=8, batch_size=128,
                      validation_split=0.1, verbose=1)
_, caps_acc = capsnet.evaluate(X_te_c, Y_te_c, verbose=0)
print(f"\n CapsNet Test Accuracy: {caps_acc*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h_caps.history['accuracy'],     lw=2, label='Train')
axes[0].plot(h_caps.history['val_accuracy'], lw=2, label='Val')
axes[0].set_title('CapsNet Accuracy', fontweight='bold'); axes[0].legend()
axes[1].plot(h_caps.history['loss'],     lw=2, color='orange', label='Train')
axes[1].plot(h_caps.history['val_loss'], lw=2, color='red',    label='Val')
axes[1].set_title('CapsNet Loss', fontweight='bold'); axes[1].legend()
plt.suptitle(f'Capsule Network on MNIST  Acc: {caps_acc*100:.2f}%', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04x_capsnet.png', dpi=130, bbox_inches='tight')
plt.show()

### Deep Belief Networks (DBNs)  Stacked RBMs on MNIST
A Deep Belief Network stacks Restricted Boltzmann Machines (RBMs).
Each RBM learns a layer of features unsupervised; a final classifier
is added on top. This cell uses sklearn's BernoulliRBM.

In [ ]:
#  DBN: Stacked RBMs + Logistic Regression 
from sklearn.neural_network import BernoulliRBM
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

(X_tr_d, y_tr_d), (X_te_d, y_te_d) = mnist.load_data()
X_tr_d = X_tr_d.reshape(-1, 784).astype('float32') / 255.0
X_te_d = X_te_d.reshape(-1, 784).astype('float32') / 255.0

# Subset for speed
N_DBN = 10000
X_tr_sub = X_tr_d[:N_DBN]; y_tr_sub = y_tr_d[:N_DBN]

#  Layer 1 RBM 
rbm1 = BernoulliRBM(n_components=256, learning_rate=0.01,
                     n_iter=15, random_state=42, verbose=0)
#  Layer 2 RBM 
rbm2 = BernoulliRBM(n_components=128, learning_rate=0.01,
                     n_iter=15, random_state=42, verbose=0)
#  Classifier 
logistic = LogisticRegression(max_iter=200, solver='saga', random_state=42)

dbn = Pipeline([('rbm1', rbm1), ('rbm2', rbm2), ('logistic', logistic)])
dbn.fit(X_tr_sub, y_tr_sub)
dbn_acc = accuracy_score(y_te_d, dbn.predict(X_te_d))
print(f" DBN (RBM2 + LR) Test Accuracy: {dbn_acc*100:.2f}%")

# Baseline: raw pixels + logistic
lr_raw = LogisticRegression(max_iter=200, solver='saga', random_state=42)
lr_raw.fit(X_tr_sub, y_tr_sub)
raw_acc = accuracy_score(y_te_d, lr_raw.predict(X_te_d))
print(f"   Baseline (raw pixels + LR): {raw_acc*100:.2f}%")

#  Visualize learned RBM filters 
fig, axes = plt.subplots(4, 16, figsize=(16, 5))
fig.suptitle(f'DBN  RBM Layer 1: 64 of 256 Learned Filters  |  Test Acc: {dbn_acc*100:.2f}%',
             fontsize=12, fontweight='bold')
for ax, component in zip(axes.ravel(), rbm1.components_[:64]):
    ax.imshow(component.reshape(28, 28), cmap='viridis', interpolation='nearest')
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04x_dbn_filters.png', dpi=130, bbox_inches='tight')
plt.show()

df_dbn = pd.DataFrame({'Model':          ['Raw Pixels + LR', 'DBN (RBM2 + LR)'],
                        'Test Accuracy':  [f'{raw_acc*100:.2f}%', f'{dbn_acc*100:.2f}%'],
                        'Feature Source': ['Raw 784 pixels', 'Unsupervised RBM features']})
print(df_dbn.to_string(index=False))